In [16]:
import os
import re
import time
import math
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

PROCESSED_DIR = "../data/processed"
RAW_DIR = "../data/raw"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

stemmer = PorterStemmer()
STANDARD_STOP = set(stopwords.words('english'))
CUSTOM_STOP_STEMMED = {
    'paper', 'propos', 'method', 'approach', 'result',
    'show', 'base', 'also', 'howev', 'therefor', 'thu',
    'furthermor', 'present', 'work', 'studi', 'experiment', 'evalu'
}
PATTERN_CLEAN = re.compile(r'[^a-z0-9\s]')
PATTERN_SPACE = re.compile(r'\s+')

def preprocess_query(text: str) -> list:
    text = str(text).lower()
    text = PATTERN_CLEAN.sub(' ', text)
    text = PATTERN_SPACE.sub(' ', text).strip()
    tokens = [t for t in text.split() if t not in STANDARD_STOP and len(t) > 2]
    stemmed = [stemmer.stem(t) for t in tokens]
    return [t for t in stemmed if t not in CUSTOM_STOP_STEMMED]

print("Da tai xong cac thu vien va cau hinh tien xu ly.")

Da tai xong cac thu vien va cau hinh tien xu ly.


In [17]:
print("Dang nap du lieu tu file Parquet...")

corpus_df = pd.read_parquet(f"{PROCESSED_DIR}/corpus_bm25.parquet")
doc_ids = corpus_df['id'].astype(str).tolist()

queries_df = pd.read_parquet(f"{RAW_DIR}/queries.parquet")
queries_dict = dict(zip(queries_df['_id'].astype(str), queries_df['text']))

qrels_df = pd.read_parquet(f"{RAW_DIR}/qrels.parquet")
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row['query-id'])
    did = str(row['corpus-id'])
    score = int(row['score'])
    if qid not in qrels_dict:
        qrels_dict[qid] = {}
    qrels_dict[qid][did] = score

print(f"Nap du lieu hoan tat.")
print("-" * 50)

print("Dang tach tu cho toan bo Corpus...")
tokenized_corpus = [str(text).split() for text in corpus_df['text_bm25']]
valid_qids = [qid for qid in qrels_dict.keys() if qid in queries_dict]

total_tokens = sum(len(doc) for doc in tokenized_corpus)
print(f"Tong so tai lieu            : {len(tokenized_corpus):,}")
print(f"Tong so token da xu ly      : {total_tokens:,}")
print(f"Chieu dai trung binh tai lieu: {total_tokens / len(tokenized_corpus):.2f} tokens")
print("-" * 50)

Dang nap du lieu tu file Parquet...
Nap du lieu hoan tat.
--------------------------------------------------
Dang tach tu cho toan bo Corpus...
Tong so tai lieu            : 25,657
Tong so token da xu ly      : 2,894,553
Chieu dai trung binh tai lieu: 112.82 tokens
--------------------------------------------------


In [18]:
from bm25_model import CustomBM25

In [19]:
def precision_at_k(retrieved, relevant, k):
    hits = sum(1 for d in retrieved[:k] if d in relevant and relevant[d] > 0)
    return hits / k

def recall_at_k(retrieved, relevant, k):
    total_rel = sum(1 for v in relevant.values() if v > 0)
    if total_rel == 0: return 0.0
    hits = sum(1 for d in retrieved[:k] if d in relevant and relevant[d] > 0)
    return hits / total_rel

def ndcg_at_k(retrieved, relevant, k):
    def dcg(ranked_list):
        return sum(relevant.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked_list[:k]))
    actual_dcg = dcg(retrieved)
    ideal_list = sorted(relevant.keys(), key=lambda x: relevant[x], reverse=True)
    ideal_dcg = dcg(ideal_list)
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

def average_precision(retrieved, relevant):
    hits = 0
    ap = 0.0
    total_rel = sum(1 for v in relevant.values() if v > 0)
    if total_rel == 0: return 0.0
    for i, d in enumerate(retrieved):
        if d in relevant and relevant[d] > 0:
            hits += 1
            ap += hits / (i + 1)
    return ap / total_rel

def reciprocal_rank(retrieved, relevant):
    for i, d in enumerate(retrieved):
        if d in relevant and relevant[d] > 0:
            return 1.0 / (i + 1)
    return 0.0

In [5]:
k1_values = [1.0, 1.2, 1.5, 1.8, 2.0, 2.2, 2.4, 2.5, 2.7, 3.0]
b_values = [0.5, 0.75, 0.85, 0.9, 0.95, 1.0]

best_map = 0.0
best_params = {'k1': 1.5, 'b': 0.9}

print("Dang chay Grid Search...")
for k1 in k1_values:
    for b in b_values:
        bm25_test = CustomBM25(tokenized_corpus, k1=k1, b=b)
        map_scores = []
        for qid in valid_qids:
            query_text = queries_dict[qid]
            relevant_docs = qrels_dict[qid]
            tokenized_query = preprocess_query(query_text)
            scores = bm25_test.get_scores(tokenized_query)
            top_indices = np.argsort(scores)[::-1][:20]
            retrieved_docs = [doc_ids[idx] for idx in top_indices]
            map_scores.append(average_precision(retrieved_docs, relevant_docs))
            
        current_map = np.mean(map_scores)
        print(f"[k1={k1:<3} | b={b:<4}] MAP: {current_map:.4f}")
        if current_map > best_map:
            best_map = current_map
            best_params = {'k1': k1, 'b': b}

print(f"Tham so tot nhat la: k1={best_params['k1']}, b={best_params['b']}")

Dang chay Grid Search...
[k1=1.0 | b=0.5 ] MAP: 0.0973
[k1=1.0 | b=0.75] MAP: 0.0993
[k1=1.0 | b=0.85] MAP: 0.1004
[k1=1.0 | b=0.9 ] MAP: 0.1005
[k1=1.0 | b=0.95] MAP: 0.1011
[k1=1.0 | b=1.0 ] MAP: 0.1010
[k1=1.2 | b=0.5 ] MAP: 0.0989
[k1=1.2 | b=0.75] MAP: 0.1010
[k1=1.2 | b=0.85] MAP: 0.1018
[k1=1.2 | b=0.9 ] MAP: 0.1020
[k1=1.2 | b=0.95] MAP: 0.1022
[k1=1.2 | b=1.0 ] MAP: 0.1022
[k1=1.5 | b=0.5 ] MAP: 0.0996
[k1=1.5 | b=0.75] MAP: 0.1025
[k1=1.5 | b=0.85] MAP: 0.1029
[k1=1.5 | b=0.9 ] MAP: 0.1028
[k1=1.5 | b=0.95] MAP: 0.1037
[k1=1.5 | b=1.0 ] MAP: 0.1038
[k1=1.8 | b=0.5 ] MAP: 0.1010
[k1=1.8 | b=0.75] MAP: 0.1031
[k1=1.8 | b=0.85] MAP: 0.1041
[k1=1.8 | b=0.9 ] MAP: 0.1044
[k1=1.8 | b=0.95] MAP: 0.1046
[k1=1.8 | b=1.0 ] MAP: 0.1044
[k1=2.0 | b=0.5 ] MAP: 0.1014
[k1=2.0 | b=0.75] MAP: 0.1035
[k1=2.0 | b=0.85] MAP: 0.1046
[k1=2.0 | b=0.9 ] MAP: 0.1050
[k1=2.0 | b=0.95] MAP: 0.1048
[k1=2.0 | b=1.0 ] MAP: 0.1050
[k1=2.2 | b=0.5 ] MAP: 0.1019
[k1=2.2 | b=0.75] MAP: 0.1036
[k1=2.2 | b=0.8

In [20]:
print("Dang khoi tao mo hinh...")

bm25_final = CustomBM25(tokenized_corpus, k1=2.5, b=0.95)

with open(f"{MODELS_DIR}/bm25_model.pkl", "wb") as f:
    pickle.dump(bm25_final, f)
print(f"Xuat file model.pkl hoan tat.")

# Luu danh sach ID dang CSV
df_doc_ids = pd.DataFrame({'doc_id': doc_ids})
df_doc_ids.to_csv(f"{MODELS_DIR}/bm25_doc_ids.csv", index=False)

print(f"Da luu thanh cong bm25_model.pkl va bm25_doc_ids.csv tai thu muc {MODELS_DIR}")
print("-" * 50)

Dang khoi tao mo hinh...
Xuat file model.pkl hoan tat.
Da luu thanh cong bm25_model.pkl va bm25_doc_ids.csv tai thu muc ../models
--------------------------------------------------


In [21]:
k_values = [5, 10, 20]
results = {k: {'P': [], 'R': [], 'NDCG': []} for k in k_values}
latencies = []
map_scores = []
rr_scores = []

print("Dang danh gia mo hinh tren tap Test...")
for qid in tqdm(valid_qids):
    query_text = queries_dict[qid]
    relevant_docs = qrels_dict[qid]
    
    t0 = time.time()
    tokenized_query = preprocess_query(query_text)
    
    # Goi phuong thuc get_scores tu mo hinh
    scores = bm25_final.get_scores(tokenized_query)
    
    # Vet 100 ket qua de tinh MAP va MRR chinh xac nhat
    top_indices = np.argsort(scores)[::-1][:100]
    retrieved_docs = [doc_ids[idx] for idx in top_indices]
    
    latencies.append(time.time() - t0)
    
    # Tinh AP va RR cho tung cau
    map_scores.append(average_precision(retrieved_docs, relevant_docs))
    rr_scores.append(reciprocal_rank(retrieved_docs, relevant_docs))
    
    for k in k_values:
        results[k]['P'].append(precision_at_k(retrieved_docs, relevant_docs, k))
        results[k]['R'].append(recall_at_k(retrieved_docs, relevant_docs, k))
        results[k]['NDCG'].append(ndcg_at_k(retrieved_docs, relevant_docs, k))

# Chot so trung binh tong the
final_map = np.mean(map_scores)
final_mrr = np.mean(rr_scores)

# Tao bang DataFrame chi chua cac chi so phu thuoc vao k
eval_data = []
for k in k_values:
    eval_data.append({
        'method': 'CustomBM25',
        'k': k,
        'P@k': round(np.mean(results[k]['P']), 4),
        'R@k': round(np.mean(results[k]['R']), 4),
        'NDCG@k': round(np.mean(results[k]['NDCG']), 4)
    })
    
df_eval = pd.DataFrame(eval_data)
df_eval.to_csv(f"{MODELS_DIR}/bm25_eval_metrics.csv", index=False)

# In ket qua ra man hinh
print("\n" + "="*55)
print(" "*12 + "KẾT QUẢ ĐÁNH GIÁ - CUSTOM BM25")
print("="*55)
print(df_eval.to_string(index=False))
print("-" * 55)
print("CÁC CHỈ SỐ TỔNG THỂ (GLOBAL METRICS):")
print(f"MAP (Mean Average Precision) : {final_map:.4f}")
print(f"MRR (Mean Reciprocal Rank)   : {final_mrr:.4f}")
print("="*55)
print(f"Thời gian phản hồi trung bình: {np.mean(latencies)*1000:.2f} ms / câu")
print(f"Đã lưu bảng chỉ số vào: {MODELS_DIR}/bm25_eval_metrics.csv")

Dang danh gia mo hinh tren tap Test...


100%|██████████| 1000/1000 [00:08<00:00, 115.06it/s]


            KẾT QUẢ ĐÁNH GIÁ - CUSTOM BM25
    method  k    P@k    R@k  NDCG@k
CustomBM25  5 0.1202 0.1218  0.1382
CustomBM25 10 0.0862 0.1746  0.1668
CustomBM25 20 0.0570 0.2310  0.1909
-------------------------------------------------------
CÁC CHỈ SỐ TỔNG THỂ (GLOBAL METRICS):
MAP (Mean Average Precision) : 0.1142
MRR (Mean Reciprocal Rank)   : 0.3010
Thời gian phản hồi trung bình: 8.58 ms / câu
Đã lưu bảng chỉ số vào: ../models/bm25_eval_metrics.csv
